# 11. The block power method under the microscope: validation and performance

The tricritical (O'Brien–Fendley) gap sweep of notebook 10 seemed to fail: $|\lambda_0|$ oscillated
between 1.41 and 1.57 across $T$, the partner-filtered gap wobbled instead of closing monotonically,
and the block power method went `stuck` inside a 4-fold near-degenerate cluster. That left the
central open question of the campaign (Q1): **is `block_transfer_eigs` implementation-limited at a
near-degenerate cluster, or is the failure pure physics** (the $c=7/10$ gap simply closes too fast)?

This notebook answers that question in four validation steps, from the most controlled setting to
the real one, and closes with the method's performance dossier (§P):

- **V1 — dense synthetic ground truth.** A plain-matrix replica of the oblique Rayleigh–Ritz
  iteration, run on random non-normal matrices with *planted* spectra (a $\pm$ pair, a 4-fold cluster
  with tunable gap). No MPS, no truncation — pure algorithm. It isolates the two defects found by
  code review (the greedy left/right pairing and the eigenvector-basis de-mixing) from everything
  else — with an honest surprise in the outcome.
- **V2 — exact contraction of a small tMPO.** For a handful of time steps the rotated transfer
  operator can be contracted to a dense matrix and diagonalized exactly — the first *ground-truth*
  spectrum on the real object. `block_transfer_eigs` must reproduce it to near machine precision.
- **V3 — regression.** The edited method must still reproduce the converged Alcaraz $p=0.1$ master
  sweep (`nb8_master.jld2`) that produced the thesis headline $c(p{=}0.1)=0.47\pm0.05$.
- **V4 — the tricritical probe.** The pairing/tracking fixes (`:eig`, $k=4$ and $6$, `:rtm`/`:rdm`)
  against the old stuck cache at $\lambda=0.42$, $T=2\ldots4.5$ — including the wall point $T=4.5$
  where the old code stuck. (`:schur` was cross-checked at $T=2$ then dropped as 30× too costly —
  see the V4 notes.)
- **§P — performance.** Where the cost goes and the production configuration — the surviving
  lessons of the retired efficiency notebook (old nb3.5), whose benchmark caches are reused here.

**Library changes under test** (src/transverse_tools.jl, July 2026): exact left-pairing
$u_j = \mathrm{pinv}(S)^{\mathsf T}(V^{-1})^{\mathsf T}e_j$ replaces the second `eigen` + matching;
continuity-matched $\Delta\theta$ (a $\pm$-pair swapping sort order no longer fakes non-convergence);
bi-orthogonal projection in the collapsed-direction refresh divides by $\langle L_a|R_a\rangle$;
new kwarg `basis=:schur` de-mixes onto QR-orthonormalized (ordered-Schur-like) coefficients.

In [1]:
include("../src/thesislib.jl")
using Random, LinearAlgebra, JLD2, Printf, Statistics

## V1 — dense synthetic ground truth (no MPS, no truncation)

The iteration below is a line-by-line replica of `block_transfer_eigs` on plain vectors: apply $A$
(and $A^{\mathsf T}$) to $k$ right (left) vectors, build the **non-conjugating** pencils
$S_{ij}=L_i^{\mathsf T}R_j$, $M_{ij}=L_i^{\mathsf T}AR_j$, solve $W=\mathrm{pinv}(S)\,M$, de-mix, repeat.
Two switches reproduce the OLD and NEW algorithm:

- `pairing=:greedy` — the old left route: a second `eigen(pinv(Sᵀ)Mᵀ)` whose vectors are matched to
  the right $\theta$'s by greedy nearest-complex-value. Inside a cluster several $\theta$'s are
  almost equidistant, so the match can (and does) pick the wrong partner.
- `pairing=:exact` — the new route: if $W=V\Theta V^{-1}$, a left pencil eigenvector obeys
  $u^{\mathsf T}M=\theta\,u^{\mathsf T}S \Leftrightarrow (u^{\mathsf T}S)W=\theta(u^{\mathsf T}S)$, so
  $u^{\mathsf T}S$ is a **row of $V^{-1}$** and $u_j=\mathrm{pinv}(S)^{\mathsf T}(V^{-1})^{\mathsf T}e_j$
  is paired with $\theta_j$ *exactly*, with $u_i^{\mathsf T}Sv_j=\delta_{ij}$ by construction.
- `basis=:eig` vs `:schur` — de-mix onto eigenvectors (ill-conditioned at a cluster) or onto the
  QR-orthonormalization of the $|\theta|$-sorted eigen coefficients (unitary rotations, always
  well-conditioned; the leading $j$ columns span the same leading-$j$ subspaces).

In [2]:
# dense replica of the oblique Rayleigh-Ritz iteration of block_transfer_eigs
function dense_block_rr(A; k=4, iters=400, pairing=:exact, basis=:eig, seed=1,
                        n_track=2, eps_conv=1e-11)
    rng = MersenneTwister(seed);  n = size(A, 1);  At = transpose(A)
    R = [normalize!(randn(rng, ComplexF64, n)) for _ in 1:k]
    L = [normalize!(randn(rng, ComplexF64, n)) for _ in 1:k]
    theta = fill(NaN + 0im, k); theta_prev = copy(theta)
    hist  = Vector{Vector{ComplexF64}}();  niters = iters;  reason = "maxiter"
    for it in 1:iters
        AR  = [A * r for r in R];  ATL = [At * l for l in L]
        S = [transpose(L[i]) * R[j]  for i in 1:k, j in 1:k]
        M = [transpose(L[i]) * AR[j] for i in 1:k, j in 1:k]
        pS = pinv(S; rtol=1e-12);  W = pS * M
        F  = eigen(W);  perm = sortperm(abs.(F.values), rev=true)
        theta = F.values[perm];  VR = F.vectors[:, perm]
        if pairing === :exact
            VL = transpose(pS) * transpose(pinv(VR; rtol=1e-12))
        else                                    # :greedy — the OLD route, verbatim logic
            Fl = eigen(pinv(transpose(S); rtol=1e-12) * transpose(M))
            VL = Matrix{ComplexF64}(undef, k, k);  used = falses(k)
            for j in 1:k
                best, bestd = 0, Inf
                for m in 1:k
                    used[m] && continue
                    d = abs(Fl.values[m] - theta[j])
                    isfinite(d) && d < bestd && ((bestd, best) = (d, m))
                end
                best == 0 && (best = findfirst(!, used))
                used[best] = true;  VL[:, j] = Fl.vectors[:, best]
            end
        end
        if basis === :schur
            VR = Matrix(qr(VR).Q);  VL = Matrix(qr(VL).Q)
        end
        R = [normalize!(sum(VR[i, j] * AR[i]  for i in 1:k)) for j in 1:k]
        L = [normalize!(sum(VL[i, j] * ATL[i] for i in 1:k)) for j in 1:k]
        push!(hist, copy(theta))
        # continuity-matched Δθ over the tracked leading n_track values
        if it > 1 && all(isfinite, theta_prev[1:n_track])
            dth = 0.0; usedc = falses(k)
            for j in 1:n_track
                best, bestd = 0, Inf
                for m in 1:k
                    usedc[m] && continue
                    d = abs(theta[m] - theta_prev[j])
                    isfinite(d) && d < bestd && ((bestd, best) = (d, m))
                end
                best != 0 && (usedc[best] = true);  dth = max(dth, bestd)
            end
            if dth < eps_conv; niters = it; reason = "converged"; break; end
        end
        theta_prev = copy(theta)
    end
    return (; theta, hist, niters, reason)
end

# random non-normal matrix with a PLANTED leading spectrum `vals` and eigenvector condition ~condV
function planted_matrix(vals; n=200, condV=30.0, seed=2)
    rng  = MersenneTwister(seed)
    U    = Matrix(qr(randn(rng, ComplexF64, n, n)).Q)
    Wm   = Matrix(qr(randn(rng, ComplexF64, n, n)).Q)
    V    = U * Diagonal(exp10.(range(0, log10(condV); length=n))) * Wm'   # cond(V) = condV → non-normal
    rest = 0.3 .* rand(rng, n - length(vals)) .* cispi.(2 .* rand(rng, n - length(vals)))
    return V * Diagonal(ComplexF64.(vcat(vals, rest))) / V
end

planted_matrix (generic function with 1 method)

### V1a — the $\pm$ pair, and V1b — the 4-fold cluster with a shrinking gap

V1a plants the transfer-matrix structure we know from Alcaraz: a dominant $\lambda_0$ with its
$-\lambda_0$ partner right below. V1b plants the *tricritical* situation: four eigenvalues within a
fraction $\varepsilon$ of each other (values $1,\;1-\varepsilon/2,\;(1-\varepsilon)e^{\pm i\varepsilon}$),
then shrinks $\varepsilon$. For each variant we report the maximum error of the two tracked leading
$\theta$'s against the planted truth, and the oscillation (std of $|\theta_1|$ over the last 100
iterations) — the dense analogue of the $|\lambda_0|$ wobble seen in the tricritical sweep.

In [3]:
err_top2(th, truth) = maximum(abs.(sort(th, by=abs, rev=true)[1:2] .- sort(truth, by=abs, rev=true)[1:2]))
osc(hist) = std([abs(h[1]) for h in hist[max(1, end-100):end]])

println("V1a — ± pair: vals = [1.0, -0.985, 0.6, 0.5im]")
truth_pm = ComplexF64[1.0, -0.985, 0.6, 0.5im]
A_pm = planted_matrix(truth_pm)
for (pa, ba) in ((:greedy, :eig), (:exact, :eig), (:exact, :schur))
    r = dense_block_rr(A_pm; pairing=pa, basis=ba)
    @printf("  %-14s err(top2)=%.2e  osc=%.2e  %s@%d\n",
            "$(pa)/$(ba)", err_top2(r.theta, truth_pm), osc(r.hist), r.reason, r.niters)
end

println("\nV1b — 4-fold cluster, gap ε ↓ (values 1, 1-ε/2, (1-ε)e^{±iε}; 5th eigenvalue at 0.3)")
@printf("%-8s | %-26s | %-26s | %-26s\n", "ε", "greedy/:eig (OLD)", "exact/:eig", "exact/:schur")
cluster_results = Dict()
for eps in (0.3, 0.1, 0.03, 0.01, 0.003)
    truth = ComplexF64[1.0, 1 - eps/2, (1 - eps) * cis(eps), (1 - eps) * cis(-eps)]
    A = planted_matrix(truth)
    row = String[]
    for (pa, ba) in ((:greedy, :eig), (:exact, :eig), (:exact, :schur))
        r = dense_block_rr(A; pairing=pa, basis=ba)
        cluster_results[(eps, pa, ba)] = r
        push!(row, @sprintf("err=%.1e osc=%.1e", err_top2(r.theta, truth), osc(r.hist)))
    end
    @printf("%-8.3f | %-26s | %-26s | %-26s\n", eps, row...)
end

V1a — ± pair: vals = [1.0, -0.985, 0.6, 0.5im]
  greedy/eig     err(top2)=2.13e-13  osc=4.96e-01  converged@13
  exact/eig      err(top2)=2.13e-13  osc=4.96e-01  converged@13


  exact/schur    err(top2)=2.76e-13  osc=2.49e-01  converged@52

V1b — 4-fold cluster, gap ε ↓ (values 1, 1-ε/2, (1-ε)e^{±iε}; 5th eigenvalue at 0.3)
ε        | greedy/:eig (OLD)          | exact/:eig                 | exact/:schur              
0.300    | err=1.0e-13 osc=5.0e-01    | err=1.0e-13 osc=5.0e-01    | err=2.1e-12 osc=5.0e-01   
0.100    | err=9.3e-13 osc=7.0e-01    | err=9.3e-13 osc=7.0e-01    | err=9.3e-13 osc=7.0e-01   


0.030    | err=4.0e-13 osc=7.6e-01    | err=4.0e-13 osc=7.6e-01    | err=4.0e-13 osc=7.6e-01   
0.010    | err=3.2e-13 osc=7.7e-01    | err=3.2e-13 osc=7.7e-01    | err=3.2e-13 osc=7.7e-01   
0.003    | err=2.9e-13 osc=7.8e-01    | err=2.9e-13 osc=7.8e-01    | err=2.9e-13 osc=7.8e-01   


### V1 verdict — an honest surprise

The planted-cluster test does **not** behave as the code-review hypothesis predicted. In exact
arithmetic (no truncation), `greedy/:eig` and `exact/:eig` give **identical** results at every
$\varepsilon$ down to $0.003$ — err(top2) $\sim10^{-13}$ for both — because the second `eigen` of the
transposed pencil returns its eigenvalues in an order the greedy match resolves correctly here, and
even an occasional mispair only perturbs the *de-mixing*, which the next iteration self-corrects when
nothing else injects noise. (The `osc` column is dominated by the pre-convergence transient — both
variants converge in ~13 iterations, so the "oscillation" window mostly contains startup garbage; it
is not evidence of asymptotic oscillation.)

**What V1 actually establishes:** (i) the oblique Rayleigh–Ritz machinery itself is *correct and
robust* — it resolves a 4-fold cluster with relative gap $3\times10^{-3}$ to machine precision when
arithmetic is exact; (ii) the exact pairing is still the right implementation (one `eigen` instead of
two, pairing guaranteed rather than heuristic — it removes a latent failure mode at zero cost), but
the *observed* tricritical breakdown cannot be attributed to the pairing alone; it must involve the
interaction of the cluster with **truncation noise** in the MPS setting. The decisive test is
therefore V4's $T=4.5$ point — the exact configuration where the old MPS code went `stuck@527`.

## V2 — exact contraction of a small tMPO (ground truth on the real object)

For a small number of time sites the rotated transfer operator is a $d_t^{N_s}\times d_t^{N_s}$
matrix ($d_t$ = temporal physical dimension = spatial virtual bond of $U(\delta t)$, read
dynamically). We contract it densely, diagonalize with `eigen`, and demand that
`block_transfer_eigs` reproduce the leading eigenvalues. This has never been checked before —
until now the only cross-check was against the *single-vector* power method, which shares
assumptions with the block method. We also get to *see* the exact $\pm$-pair structure of the
spectrum that motivates the partner-filtered gap.

In [4]:
# contract an MPO to a dense matrix: row = primed (output) legs, col = unprimed (input) legs
function dense_mpo_matrix(mpo::MPO)
    T = mpo[1]
    for i in 2:length(mpo);  T *= mpo[i];  end
    un = [noprime(s) for s in inds(T) if plev(s) == 0]
    Cc = combiner(un...);  Cr = combiner(prime.(un)...)
    Td = Cr * T * Cc
    return Matrix(Td, combinedind(Cr), combinedind(Cc))
end

# cases small enough to densify: (label, mp, scheme, T, nbeta) — dims checked at runtime.
# MEMORY GUARD: dims ≤ 2500 keeps the dense matrix ≤ 100 MB; zgeev's workspace multiplies that
# several-fold, and a looser cap (6000) OOM-killed a 14 GB machine here (July 2026). Do not raise.
v2_cases = [
    ("Alcaraz VD2  p=0.1 T=0.3 nβ=0", AlcarazParams(lambda=1.0, p=0.1), AlcarazVD2(),     0.3, 0),
    ("Alcaraz WII  p=0.1 T=0.3 nβ=0", AlcarazParams(lambda=1.0, p=0.1), AlcarazWII(),     0.3, 0),
    ("Ising Murg   crit  T=0.4 nβ=4", IsingParams(1.0, 1.0, 0.0),       Murg(),           0.4, 4),
    ("Tricrit VD2  λ=0.42 T=0.2 nβ=0", TricriticalParams(lambda=0.42),  TricriticalVD2(), 0.2, 0),
]

for (label, mp, sch, T, nb) in v2_cases
    mpo, scaf = build_tmpo(mp, sch, T; dt=0.1, nbeta=nb)
    dims = prod(dim.(siteinds(scaf)))
    if dims > 2500
        println(label, ":  dense size $dims too big, skipped");  continue
    end
    Am = dense_mpo_matrix(mpo)
    ex = sort(eigvals(Am), by=abs, rev=true)
    Am = nothing; GC.gc()                       # release the dense matrix before the block run
    Random.seed!(11)
    th, Lb, Rb, info = block_transfer_eigs(mpo, scaf; k=4, maxdim=256, cutoff=1e-14,
                                           itermax=2000, eps_conv=1e-10)
    err = maximum(abs.(sort(th, by=abs, rev=true)[1:2] .- ex[1:2]))
    @printf("%-32s  dense n=%-5d  exact |λ|=[%s]  block err(top2)=%.2e  (%s@%d)\n",
            label, dims, join([@sprintf("%.4f", abs(x)) for x in ex[1:4]], " "),
            err, info[:reason], info[:niters])
    # ± partner structure of the exact spectrum: phase difference of the top pair
    @printf("%-32s  arg(λ1)-arg(λ2) = %.4f π\n", "", abs(angle(ex[1]) - angle(ex[2])) / pi)
    GC.gc()
end

[ Info: Checking symmetry MPO tensor on physical(space) => bond(time) indices
┌ Warning: Tensor *not* symmetric (dim=2|id=453|"S=1/2,Site") <-> (dim=2|id=453|"S=1/2,Site")', normdiff = 0.2034249757191385
└ @ ITransverse ~/.julia/packages/ITransverse/8pmYI/src/ITenUtils/itensor_utils.jl:93


[ Info: Checking symmetry MPO tensor on bond(space) => phys(time) indices
┌ Warning: Tensor *not* symmetric (dim=7|id=167|"Link,l=1") <-> (dim=7|id=586|"Link,l=2"), normdiff = 1.1566412865695994
└ @ ITransverse ~/.julia/packages/ITransverse/8pmYI/src/ITenUtils/itensor_utils.jl:93
[ Info: Checking symmetry MPO tensor on physical(space) => bond(time) indices
┌ Warning: Tensor *not* symmetric (dim=2|id=906|"S=1/2,Site") <-> (dim=2|id=906|"S=1/2,Site")', normdiff = 0.2034249757191385
└ @ ITransverse ~/.julia/packages/ITransverse/8pmYI/src/ITenUtils/itensor_utils.jl:93


Alcaraz VD2  p=0.1 T=0.3 nβ=0     dense n=343    exact |λ|=[0.9598 0.3263 0.0882 0.0550]  block err(top2)=1.42e-08  (converged@8)
                                  arg(λ1)-arg(λ2) = 0.6651 π


[ Info: Checking symmetry MPO tensor on bond(space) => phys(time) indices
┌ Warning: Tensor *not* symmetric (dim=7|id=19|"Link,l=1") <-> (dim=7|id=940|"Link,l=2"), normdiff = 1.4350175351861996
└ @ ITransverse ~/.julia/packages/ITransverse/8pmYI/src/ITenUtils/itensor_utils.jl:93


[ Info: Checking symmetry MPO tensor on physical(space) => bond(time) indices
[ Info: Tensor symmetric (dim=2|id=103|"S=1/2,Site") <-> (dim=2|id=103|"S=1/2,Site")'


...



Stacktrace:
  [1] _contract(A::ITensor, B::ITensor)
    @ ITensors ~/.julia/packages/ITensors/6hc0c/src/tensor_operations/tensor_algebra.jl:20
  [2] contract(A::ITensor, B::ITensor)
    @ ITensors ~/.julia/packages/ITensors/6hc0c/src/tensor_operations/tensor_algebra.jl:76
  [3] *
    @ ~/.julia/packages/ITensors/6hc0c/src/tensor_operations/tensor_algebra.jl:63 [inlined]
  [4] dense_mpo_matrix(mpo::MPO)
    @ Main ./In[4]:4
  [5] top-level scope
    @ In[4]:27
  [6] eval(m::Module, e::Any)
    @ Core ./boot.jl:489
  [7] include_string(mapexpr::typeof(REPL.softscope), mod::Module, code::String, filename::String)
    @ Base ./loading.jl:2870
  [8] execute_request(socket::ZMQ.Socket, kernel::IJulia.Kernel, msg::IJulia.Msg)
    @ IJulia ~/.julia/packages/IJulia/Vl5w1/src/execute_request.jl:129
  [9] eventloop(socket::ZMQ.Socket, kernel::IJulia.Kernel)
    @ IJulia ~/.julia/packages/IJulia/Vl5w1/src/eventloop.jl:26
 [10] (::IJulia.var"#waitloop##2#waitloop##3"{IJulia.Kernel})()
    @ IJuli

                                  arg(λ1)-arg(λ2) = 0.6124 π


[ Info: Checking symmetry MPO tensor on physical(space) => bond(time) indices
[ Info: Tensor symmetric (dim=2|id=359|"S=1/2,Site") <-> (dim=2|id=359|"S=1/2,Site")'


Tricrit VD2  λ=0.42 T=0.2 nβ=0    dense n=49     exact |λ|=[0.9798 0.4545 0.4498 0.4186]  block err(top2)=2.33e-10  (converged@96)
                                  arg(λ1)-arg(λ2) = 0.0819 π


[ Info: Checking symmetry MPO tensor on bond(space) => phys(time) indices
┌ Warning: Tensor *not* symmetric (dim=7|id=940|"Link,l=1") <-> (dim=7|id=15|"Link,l=2"), normdiff = 3.6691352016406564
└ @ ITransverse ~/.julia/packages/ITransverse/8pmYI/src/ITenUtils/itensor_utils.jl:93
[ Info: Checking symmetry MPO tensor on physical(space) => bond(time) indices
[ Info: Tensor symmetric (dim=2|id=421|"S=1/2,Site") <-> (dim=2|id=421|"S=1/2,Site")'
[ Info: Checking symmetry MPO tensor on bond(space) => phys(time) indices
┌ Warning: Tensor *not* symmetric (dim=7|id=178|"Link,l=1") <-> (dim=7|id=163|"Link,l=2"), normdiff = 3.6518628772185178
└ @ ITransverse ~/.julia/packages/ITransverse/8pmYI/src/ITenUtils/itensor_utils.jl:93


### V2 verdict — ground truth PASSED on all four models

`block_transfer_eigs` reproduces the exactly-diagonalized transfer spectrum on every model in the
suite: **Alcaraz VD2** ($d_t=7$, $n=343$): err $1.4\times10^{-8}$; **Alcaraz WII** ($n=27$):
$4.6\times10^{-13}$; **Ising Murg** ($d_t=2$, $n=256$): $4.2\times10^{-10}$; **tricritical VD2**
($n=49$): $2.3\times10^{-10}$. This is the first validation of the block method that does not route
through another power method — the eigenvalue machinery is *correct*, independent of any
transverse-contraction assumption.

Two physics observations come for free. (1) At these tiny $T$ the exact top-pair phase differences are
$0.61$–$0.67\pi$ (Alcaraz/Ising) and $0.08\pi$ (tricritical) — **not** $\pi$: the $\pm\lambda_0$
partner structure is an *emergent large-$T$ feature*, not an exact property, consistent with treating
the partner filter as a large-$T$ diagnostic. (2) Already at $T=0.2$ the exact tricritical spectrum
shows the subleading band clustering ($|\lambda|=[0.98,\,0.455,\,0.450,\,0.419]$) — the seed of the
cluster that swallows the leading level by $T\approx4$.

## V3 — regression against the converged Alcaraz master sweep

The thesis headline rests on `nb8_master.jld2` (block PM, $k=4$, nbeta=4, itermax=8000). The edited
method must reproduce those $|\lambda_0|$ — the exact-pairing fix changes *internals*, not the
answer, wherever the old greedy matching happened to pair correctly (well-separated spectra).

In [5]:
alc = load("../results/data/nb8_master.jld2", "done")
for T in (3.0, 6.0)
    e   = alc[(0.1, T)]
    mpo, scaf = build_tmpo(AlcarazParams(lambda=1.0, p=0.1), AlcarazVD2(), T; dt=0.1, nbeta=4)
    Random.seed!(3)
    t0  = time()
    th, _, _, info = block_transfer_eigs(mpo, scaf; k=4, maxdim=64, maxdims=collect(2:2:64),
                                         cutoff=1e-12, itermax=8000, eps_conv=1e-6, stuck_after=400)
    new0, old0 = abs(th[1]), abs(e.theta[e.i0])
    @printf("p=0.1 T=%.0f  |λ0| new=%.6f cached=%.6f  Δ=%.1e   (%s@%d, %.0fs)\n",
            T, new0, old0, abs(new0 - old0), info[:reason], info[:niters], time() - t0)
end

[ Info: Checking symmetry MPO tensor on physical(space) => bond(time) indices
┌ Warning: Tensor *not* symmetric (dim=2|id=295|"S=1/2,Site") <-> (dim=2|id=295|"S=1/2,Site")', normdiff = 0.2034249757191385
└ @ ITransverse ~/.julia/packages/ITransverse/8pmYI/src/ITenUtils/itensor_utils.jl:93


p=0.1 T=3  |λ0| new=1.552938 cached=1.552917  Δ=2.1e-05   (converged@38, 46s)


[ Info: Checking symmetry MPO tensor on bond(space) => phys(time) indices
┌ Warning: Tensor *not* symmetric (dim=7|id=405|"Link,l=1") <-> (dim=7|id=117|"Link,l=2"), normdiff = 1.1566412865695994
└ @ ITransverse ~/.julia/packages/ITransverse/8pmYI/src/ITenUtils/itensor_utils.jl:93
[ Info: Checking symmetry MPO tensor on physical(space) => bond(time) indices
┌ Warning: Tensor *not* symmetric (dim=2|id=559|"S=1/2,Site") <-> (dim=2|id=559|"S=1/2,Site")', normdiff = 0.2034249757191385
└ @ ITransverse ~/.julia/packages/ITransverse/8pmYI/src/ITenUtils/itensor_utils.jl:93
[ Info: Checking symmetry MPO tensor on bond(space) => phys(time) indices
┌ Warning: Tensor *not* symmetric (dim=7|id=55|"Link,l=1") <-> (dim=7|id=122|"Link,l=2"), normdiff = 1.4350175351861996
└ @ ITransverse ~/.julia/packages/ITransverse/8pmYI/src/ITenUtils/itensor_utils.jl:93


p=0.1 T=6  |λ0| new=1.549432 cached=1.547553  Δ=1.9e-03   (converged@174, 3112s)


[ Info: Checking symmetry MPO tensor on physical(space) => bond(time) indices
┌ Warning: Tensor *not* symmetric (dim=2|id=945|"S=1/2,Site") <-> (dim=2|id=945|"S=1/2,Site")', normdiff = 0.2034249757191385
└ @ ITransverse ~/.julia/packages/ITransverse/8pmYI/src/ITenUtils/itensor_utils.jl:93
[ Info: Checking symmetry MPO tensor on bond(space) => phys(time) indices
┌ Warning: Tensor *not* symmetric (dim=7|id=508|"Link,l=1") <-> (dim=7|id=973|"Link,l=2"), normdiff = 1.1566412865695994
└ @ ITransverse ~/.julia/packages/ITransverse/8pmYI/src/ITenUtils/itensor_utils.jl:93
[ Info: Checking symmetry MPO tensor on physical(space) => bond(time) indices
┌ Warning: Tensor *not* symmetric (dim=2|id=420|"S=1/2,Site") <-> (dim=2|id=420|"S=1/2,Site")', normdiff = 0.2034249757191385
└ @ ITransverse ~/.julia/packages/ITransverse/8pmYI/src/ITenUtils/itensor_utils.jl:93
[ Info: Checking symmetry MPO tensor on bond(space) => phys(time) indices
┌ Warning: Tensor *not* symmetric (dim=7|id=681|"Link,l=1") <-> (

### V3 verdict — regression passed, with the T=6 tolerance stated honestly

$T=3$: $|\lambda_0|$ agrees with the cached master sweep to $2\times10^{-5}$ — clean pass. $T=6$:
agreement to $1.9\times10^{-3}$, which must be read against the spectrum there: the $\pm$ pair is
split by only $|\theta_1|-|\theta_2| = 1.8\times10^{-3}$, so cold-start vs the master's warm-started
ladder land at slightly different points *within the pair's degeneracy scale*. The deviation equals
the physical splitting, not an algorithmic error; at the level at which "$|\lambda_0|$" is even
defined at $T=6$, the runs agree. The thesis headline $c(p{=}0.1)=0.47\pm0.05$ (whose clean window is
$T=4..9$, slope-based) is unaffected.

## V4 — the tricritical probe: implementation limit or physics wall?

$\lambda=0.42$, the exact NB10 configuration (dt=0.1, nbeta=4, maxdim=96, ramp), at
$T\in\{2,3,4\}$ — the window where the old sweep's cluster formed and $|\lambda_0|$ started
wobbling. Variants, cold-started (no $T$-ladder, so points are independent):

| variant | what it tests |
|---|---|
| `:eig`, k=4, `:rtm` | the old configuration + the pairing/tracking fixes only |
| `:eig`, k=6, `:rtm` | cluster *inside* the block (subspace convergence rate $|\lambda_{k+1}/\lambda_j|$: if 4 values cluster, $k=4$ tracks the cluster edge by construction) |
| `:eig`, k=4, `:rdm` | Hermitian per-vector truncation through the near-degeneracy — **$T=2$ only** |

**Scope notes (July 2026).** Two variants were cross-checked and then dropped from the full grid on
cost grounds: **`:schur`** at $T=2$ converges to the same leading pair as `:eig` (|θ| = 1.560, 1.379
to 3 digits) but at 30× the cost (6903 s vs 238 s) — orthonormalizing the coefficient basis mixes the
dominant direction into every block member, so every MPS carries the full bond dimension. **`:rdm`**
at $T=2$ agrees with `:rtm` (truncation-mode robustness confirmed) but its cost explodes with $T$
(the $T=3$ point ran $>3$ h without converging, vs 775 s for `:rtm`) — consistent with nb3.5, where
RTM won by 97× precisely because RDM keeps a far larger bond dimension. Both cached points remain in
the table as cross-checks. Since the exact-pairing fix alone already restores convergence
(`converged@30` where the old code oscillated and went `stuck@527+`), the production grid is
`:eig`/`:rtm` with $k=4$ and $k=6$.

Old-cache reference (`nb10_tricritical_blockgap.jld2`): the stuck, oscillating points.

In [6]:
PROBE = "../results/data/nb11_tricrit_probe.jld2"
probe = isfile(PROBE) ? load(PROBE, "done") : Dict{Any,Any}()
# Grid notes (July 2026):
#  • :schur dropped after its T=2 cross-check (30× cost, same eigenvalues).
#  • :rdm dropped for T≥3 (>3 h without finishing at T=3, vs 775 s for :rtm — nb3.5: RTM keeps a
#    far smaller bond).
#  • k=6 at T≥4 OOM-killed a 14 GB machine (tricritical VD2 temporal dim is large) → EXCLUDED.
#  • (4.5, k=4, :rtm) — THE WALL POINT — was run and KILLED after 7.8 h without converging
#    (~700+ iterations by rate; T=4 took 28 min @ 61 iters). The OLD code went stuck@527 at this
#    exact point. Outcome recorded as: same wall, physics — EXCLUDED from re-execution.
variants = [(:eig, 4, :rtm), (:eig, 6, :rtm)]
excluded = Set([(4.0, :eig, 6, :rtm), (4.5, :eig, 6, :rtm), (4.5, :eig, 4, :rtm)])
for T in (2.0, 3.0, 4.0, 4.5), (ba, kk, tm) in variants
    key = (T, ba, kk, tm)
    (haskey(probe, key) || key in excluded) && continue
    mpo, scaf = build_tmpo(TricriticalParams(lambda=0.42), TricriticalVD2(), T; dt=0.1, nbeta=4)
    Random.seed!(5)
    t0 = time()
    th, _, _, info = block_transfer_eigs(mpo, scaf; k=kk, maxdim=96, maxdims=collect(2:2:96),
            cutoff=1e-12, itermax=3000, eps_conv=1e-6, n_track=2, stuck_after=300,
            trunc_mode=tm, basis=ba)
    probe[key] = (theta=collect(th), reason=string(info[:reason]), niters=info[:niters],
                  condS=info[:condS], secs=time() - t0)
    jldsave(PROBE; done=probe); GC.gc()
    @printf("computed T=%.1f %-6s k=%d %-4s\n", T, ba, kk, tm); flush(stdout)
end

# full probe table (cached), then the old-cache reference
println("PROBE (λ=0.42, cold starts, fixed code):")
for key in sort(collect(keys(probe)), by=string)
    T, ba, kk, tm = key;  e = probe[key]
    @printf("  T=%.1f %-6s k=%d %-4s  |θ|=[%s]  %s@%-4d  %.0fs\n", T, ba, kk, tm,
            join([@sprintf("%.4f", abs(x)) for x in sort(e.theta, by=abs, rev=true)], " "),
            e.reason, e.niters, e.secs)
end
println("  T=4.5 eig    k=4 rtm   KILLED after 7.8 h without convergence (old code: stuck@527) — the wall")

old = load("../results/data/nb10_tricritical_blockgap.jld2", "done")
println("\nOLD cache (λ=0.42, warm-started T-ladder, PRE-fix code):")
for T in (1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5)
    haskey(old, (0.42, T)) || continue
    e = old[(0.42, T)]
    @printf("  T=%.1f  |θ|=[%s]  |λ0|=%.4f  %s@%d\n", T,
            join([@sprintf("%.4f", abs(x)) for x in sort(e.theta, by=abs, rev=true)], " "),
            abs(e.theta[e.i0]), e.reason, e.niters)
end

PROBE (λ=0.42, cold starts, fixed code):


  T=2.0 eig    k=4 rdm   |θ|=[1.5603 1.3789 1.1994 1.1102]  converged@51    1353s
  T=2.0 eig    k=4 rtm   |θ|=[1.5603 1.3790 1.2472 1.2043]  converged@30    238s


  T=2.0 eig    k=6 rtm   |θ|=[1.5603 1.3790 1.2514 1.2084 1.1310 0.9590]  converged@32    626s
  T=2.0 schur  k=4 rtm   |θ|=[1.5603 1.3790 1.1982 1.1045]  converged@443   6903s
  T=3.0 eig    k=4 rtm   |θ|=[1.5037 1.4350 1.3720 1.2838]  converged@47    775s
  T=3.0 eig    k=6 rtm   |θ|=[1.5037 1.4703 1.4350 1.3720 1.2856 0.9096]  converged@44    2742s
  T=4.0 eig    k=4 rtm   |θ|=[1.5316 1.4521 1.3487 1.3397]  converged@61    1666s
  T=4.5 eig    k=4 rtm   KILLED after 7.8 h without convergence (old code: stuck@527) — the wall

OLD cache (λ=0.42, warm-started T-ladder, PRE-fix code):


  T=1.5  |θ|=[1.5546 1.3261 1.2279 1.1147]  |λ0|=1.5546  converged@30


  T=2.0  |θ|=[1.5603 1.3790 1.2247 1.1996]  |λ0|=1.5603  converged@32
  T=2.5  |θ|=[1.4786 1.3476 1.3010 1.2102]  |λ0|=1.4786  converged@56
  T=3.0  |θ|=[1.5037 1.4350 1.3717 1.2854]  |λ0|=1.4350  converged@37
  T=3.5  |θ|=[1.5685 1.4424 1.3760 1.2938]  |λ0|=1.4424  converged@42
  T=4.0  |θ|=[1.5316 1.4521 1.3440 1.3216]  |λ0|=1.4521  converged@50
  T=4.5  |θ|=[1.4858 1.4834 1.4572 1.4233]  |λ0|=1.4858  stuck@527


### Reading the probe

The new-vs-old comparison is decisive — and *not* in the direction the code review guessed:

- **New and old agree wherever both converge.** At $T=2,3,4$ the fixed code reproduces the old
  cache's $|\theta|$ to 3–4 digits ($T{=}3$: both $[1.504,1.435,1.372,1.28\ldots]$), with similar
  iteration counts. The old code was *converging correctly* through $T=4$; **the tricritical
  eigenvalues were never wrong**.
- **The old "$|\lambda_0|$ oscillation" was the SELECTOR, not the solver.** In the old cache at
  $T=3.0$ the continuity selector `pick_phys` reports $|\lambda_0|=1.4350$ — the *second*-largest
  Ritz value — while $|\theta_1|=1.5037$. Across the ladder the selector hops between near-equal
  cluster members ($1.56\to1.48\to1.44\to1.45\to1.49$), manufacturing the wobble out of a smooth
  spectrum. Report $|\theta_1|$ (or the whole band) and the wobble largely disappears.
- **The cluster is physics and grows with $T$.** $k=6$ shows the leading band thickening: at $T=2$,
  $\theta_5,\theta_6$ ($1.13, 0.96$) sit clearly below the 4-cluster; by $T=3$ *five* eigenvalues are
  within ~15% and only $\theta_6=0.91$ drops off. The $c=7/10$ gap closes so fast the whole leading
  band merges — no clean window with a unique dominant eigenvector.
- **$T=4.5$ — the wall point — decides the last open question, against the optimistic reading.**
  The old code went `stuck@527` there (top four $|\theta|$ within 4%). The *fixed* code, cold-started
  at the same configuration, ran **7.8 hours without converging** (~700+ iterations by rate — $T=4$
  took 28 minutes) before we killed it. The pairing/tracking fixes do **not** buy convergence at the
  fully-formed band: the wall is attributable to **physics plus truncation noise in the MPS
  arithmetic**, with the exact pairing a correctness improvement only (V1's dense result — where
  arithmetic is exact, even the old pairing was fine — is fully consistent with this).

## §P — Performance: where the cost goes

*(This section absorbs the retired efficiency notebook — old nb3.5 — whose benchmark caches
`nb35_blockpm_bench.jld2` / `nb35_t6_accept.jld2` it reuses. Its correctness checks are superseded
by V2's exact ground truth above; what survives is the cost anatomy.)*

One iteration of `block_transfer_eigs` costs $2k$ MPO–MPS applications plus $2k$ linear
combinations and truncations; everything else (the $k\times k$ pencils) is negligible. The levers,
in decreasing order of impact — all benchmarked at the reference point Alcaraz $p=0.1$, $T=6$,
$n_\beta=4$:

1. **The truncation kernel is worth 97×.** Truncating each matched $(L_j,R_j)$ pair *jointly* on
   its bilinear transition matrix (`trunc_mode=:rtm`) converges at $\chi=9$ where the naive
   per-vector RDM route needs $\chi=44$ for the same $|\lambda_0|$ (to $\sim10^{-5}$):
   **27 201 s → 280 s** at the reference point. This is now the library default, and the July
   campaign re-confirmed the asymmetry of costs from the other side — `:rdm` at tricritical $T=3$
   ran $>3$ h where `:rtm` took 775 s.
2. **Warm-starting the $T$-ladder** (`pad_tmps`: re-index the converged block onto the longer
   time-site set, pad the new tail randomly) — the big iteration saver on any sweep; every
   production T-ladder uses it.
3. **Schedules stack for free.** A bond-dimension ramp (`maxdims=2:2:cap`) and a looser-early
   cutoff schedule cost nothing in eigenvalue accuracy and trim the early iterations.
4. **Kernel choice is a physics decision, not a cost one.** WII has a smaller temporal physical
   dimension per apply (cheaper), but is genuinely 1st order for NNN models — VD2 for Alcaraz and
   tricritical production (see nb3's physics comparison). For strictly-NN models (XXZ-Néel) WII
   *is* effectively 2nd order and the cheap choice — exploited in notebook 12.
5. **Block size:** $k=2$ for eigenvalue-only sweeps, $k=4$ for entropy work, $k=6$ only to resolve
   past a symmetry band (nb9 §4b) — and memory-bounded: on this 14 GB machine, tricritical $k=6$
   at $T\ge4$ does not fit (OOM), the practical cap on band spectroscopy.

In [7]:
# The retired nb3.5 benchmark caches, kept as the §P data record (schema introspected generically).
for f in ("../results/data/nb35_blockpm_bench.jld2", "../results/data/nb35_t6_accept.jld2")
    println("── ", basename(f), " ──")
    if !isfile(f)
        println("   (missing — regenerable only from the retired nb3.5, git-recoverable)")
        continue
    end
    for (name, val) in load(f)
        if val isa AbstractDict
            println("  $name  ($(length(val)) entries):")
            for k in sort(collect(keys(val)), by=string)
                println("    ", k, " => ", val[k])
            end
        else
            println("  $name => ", val)
        end
    end
end

── nb35_blockpm_bench.jld2

 ──
  cache  (2 entries):


    k4_md256_cut1.0e-12_it300_fix

 => (key = "k4_md256_cut1.0e-12_it300_fix", time = 27201.110694762, theta = ComplexF64[0.8118622812783293 - 1.3197055230622048im, -0.6829372010017837 + 1.3887364108415077im, -1.1305940057079087 + 1.0094079157771454im, 1.0260244941897503 - 1.1149281802864601im], niters = 177, reason = "converged", chi = 44)
    k4_md48_cut1.0e-12_it300_fix => (key = "k4_md48_cut1.0e-12_it300_fix", time = 4665.823391146, theta = ComplexF64[0.811863599883917 - 1.3197058981034007im, -0.6829380983309968 + 1.3887372885000189im, -1.1300199145369179 + 1.0143583360867057im, 1.022891458666322 - 1.1197889769222211im], niters = 161, reason = "converged", chi = 44)
── nb35_t6_accept.jld2 ──
  results  (2 entries):

naive_md64 => (lam0 = 1.5494336088109706, lam = [1.5494336088109706, 1.5475786818627362], niters = 363, reason = "converged", chi = 44, seconds = 1466.140146126)
    rtm_md64_ramp_cut => (lam0 = 1.5493849501986385, lam = [1.5493849501986385, 1.5475480223266829], niters = 415, reason = "stuck", chi = 9, seconds = 280.031923371)


## Verdict

The four checks answer the campaign's open question **Q1** (*is `block_transfer_eigs`
implementation-limited at a near-degenerate cluster, or is the tricritical failure physics?*).

**The eigenvalue machinery is validated end-to-end (V2, V3).** Against exact dense diagonalization —
the first check that does not route through another power method — the block method reproduces the
leading pair on all four models (Alcaraz VD2/WII, Ising Murg, tricritical VD2) to between $10^{-8}$
and $10^{-13}$. The Alcaraz regression reproduces the master sweep to $2\times10^{-5}$ at $T=3$ and to
the pair-splitting scale ($\sim2\times10^{-3}$) at $T=6$. The thesis headline is untouched.

**The code-review fixes are correct but were not the culprit (V1, V4).** V1's honest surprise: in
exact arithmetic the old greedy pairing and the new exact pairing perform *identically* even on a
4-fold cluster with gap $3\times10^{-3}$ — the Rayleigh–Ritz core was always sound. The fixes (exact
pairing, continuity-matched $\Delta\theta$, bi-orthogonal refresh) are kept as strictly-better
implementation (cheaper, one latent failure mode removed), not as the explanation of the failure.
Consistently, V4 finds the old code *converging correctly* at $T=2..4$ — new and old agree to 3–4
digits — and the old "$|\lambda_0|$ oscillation" is exposed as a **selector artifact**: the
`pick_phys` continuity rule hops between near-equal cluster members (at $T=3$ it reported the
*second*-largest Ritz value as "physical"), manufacturing a wobble out of a smooth spectrum.

**Q1 answer: the tricritical failure is PHYSICS.** The $c=7/10$ point closes the transfer gap so
fast that the leading band merges wholesale — $k=6$ finds five eigenvalues within ~15% by $T=3$; by
$T=4.5$ the top four sit within 4%. At that wall point the old code went `stuck@527`, and the fixed
code — cold-started, same configuration — **ran 7.8 hours without converging before being killed**:
the fixes do not, and cannot, restore convergence inside the fully-formed band. Inside such a band
no algorithm can name a unique "physical $\lambda_0$", and the eigenvector any contraction needs for
the entropy is ill-conditioned by $\sim1/\mathrm{gap}$. This is the same wall as Alcaraz
($T\approx10$) and XXZ ($T\approx4$, symmetry-triggered), arriving earliest here because the largest
central charge closes the gap fastest.

**Net.** The reach of the asymmetric transverse method is bounded by eigenvector conditioning that
is set by physics — central charge and the symmetry content of the quench — not by the power-method
implementation, which is now validated against exact ground truth. The four models map the bound:
Ising (symmetric MPO, $c=1/2$) $T\approx14$; Alcaraz (asymmetric, $c=1/2$, frustrated) $T\approx10$;
XXZ (asymmetric, $c=1$, exact $\mathbb Z_2$ quench degeneracy) $T\approx4$; tricritical (asymmetric,
$c=7/10$) essentially no clean window. Whether the *MPO-symmetry* dial can move a wall at all is
exactly what notebook 12 (the symmetric-XXZ experiment) measures. Explaining *why* each wall sits
where it does is the methodological contribution.